# P10.6-AI — Notebook 63 v2: entrenamiento subarticular Axial T2

Flujo resistente a cortes y cambios de cuenta.

**Fase A — CPU:** ejecutar `1 → 2 → 3A`. La celda 3A guarda cada `.npy` completo directamente en Drive y puede retomarse.

**Fase B — GPU:** cambiar a T4 o superior y volver a ejecutar `1 → 2 → 3B → 4 → 5`. La celda 3B copia el caché completo al SSD local.

No ejecutar 3A desde dos runtimes al mismo tiempo. El `internal_test` permanece sellado para el Notebook 64.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


In [ ]:
# 1) Dependencias, Drive y pipeline
from __future__ import annotations

import getpass
import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive  # type: ignore

packages = {
    "pydicom": "pydicom",
    "timm": "timm",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "kaggle": "kaggle",
}
missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *missing
    ])

print({
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch": torch.__version__,
    "installedNow": missing,
})
drive.mount("/content/drive", force_remount=False)

REPO_URL = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)
REPO_REF = "enzo/p10-6-ai-rsna-findings"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_ROOT),
    ])
else:
    subprocess.check_call(
        ["git", "fetch", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "checkout", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "pull", "--ff-only", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )

REPO_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()

ai_service_path = str(REPO_ROOT / "ai_service")
if ai_service_path not in sys.path:
    sys.path.insert(0, ai_service_path)

from pfi_ai_service.training.rsna_subarticular_training import (
    TrainConfig,
    build_cache,
    download_selected_series,
    find_data_root,
    load_manifests,
    prepare_samples,
    train_model,
)

print({"repoRef": REPO_REF, "repoSha": REPO_SHA})


In [ ]:
# 2) Rutas, configuración, manifests y DICOM
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
if not PFI_ROOT.is_dir():
    raise RuntimeError(
        "No se encontró PFI_MVP en Mi unidad. "
        "Agregá la carpeta compartida como acceso directo o corregí PFI_ROOT."
    )

RESULTS_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings"
SPLIT_ROOT = RESULTS_ROOT / "notebook62_subarticular_split"
RUN_ROOT = RESULTS_ROOT / "notebook63_subarticular_training"
MODEL_ROOT = (
    PFI_ROOT / "models" / "P10_6_rsna_findings"
    / "subarticular_axial_t2_2p5d"
)
CHECKPOINT_ROOT = MODEL_ROOT / "checkpoints"

LOCAL_DATA_ROOT = Path("/content/RSNA_LUMBAR_DISC")
DRIVE_DATA_ROOT = PFI_ROOT / "data" / "RSNA_LUMBAR_DISC"
PERSISTENT_CACHE_ROOT = (
    PFI_ROOT / "cache" / "notebook63_subarticular_cache"
)
LOCAL_CACHE_ROOT = Path("/content/rsna_subarticular_cache")
COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"

CFG = TrainConfig(
    seed=2026,
    image_size=224,
    crop_size=256,
    batch_size=32,
    num_workers=2,
    max_epochs=15,
    patience=5,
    learning_rate=2e-4,
    weight_decay=1e-4,
    model_name="efficientnet_b0",
    pretrained=True,
    minimum_macro_f1=0.36,
    minimum_balanced_accuracy=0.45,
    minimum_severe_recall=0.30,
    minimum_moderate_recall=0.25,
)

for path in (RUN_ROOT, MODEL_ROOT, CHECKPOINT_ROOT, PERSISTENT_CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

train_manifest, validation_manifest, split_summary, manifest_hashes = (
    load_manifests(SPLIT_ROOT)
)

data_root, data_audits = find_data_root(
    [LOCAL_DATA_ROOT, DRIVE_DATA_ROOT],
    train_manifest,
    validation_manifest,
)
print({
    "trainRows": len(train_manifest),
    "validationRows": len(validation_manifest),
    "dataAudits": [
        {
            "root": audit.root,
            "complete": audit.complete,
            "missingSeries": audit.missing_series,
        }
        for audit in data_audits
    ],
    "internalTestAccessed": False,
})

if data_root is None:
    kaggle_token = getpass.getpass("Kaggle API token: ")
    data_root = download_selected_series(
        train_manifest,
        validation_manifest,
        LOCAL_DATA_ROOT,
        COMPETITION,
        kaggle_token,
    )

print({
    "selectedDataRoot": str(data_root),
    "persistentCacheRoot": str(PERSISTENT_CACHE_ROOT),
    "checkpointRoot": str(CHECKPOINT_ROOT),
})


## 3A — CPU: construir o reanudar caché persistente

Esta celda reutiliza los `.npy` existentes. Para pausar: detener manualmente 3A y ejecutar la celda **3A-checkpoint**.


In [ ]:
# 3A) Construir o reanudar caché persistente
train_samples = prepare_samples(train_manifest, data_root, "train")
validation_samples = prepare_samples(
    validation_manifest,
    data_root,
    "validation",
)

for temporary in PERSISTENT_CACHE_ROOT.rglob("*.tmp.npy"):
    temporary.unlink(missing_ok=True)

train_cache_audit = build_cache(
    train_samples,
    PERSISTENT_CACHE_ROOT,
    "train",
    CFG,
)
validation_cache_audit = build_cache(
    validation_samples,
    PERSISTENT_CACHE_ROOT,
    "validation",
    CFG,
)

cache_report = {
    "cacheRoot": str(PERSISTENT_CACHE_ROOT),
    "train": {
        "present": len(list(
            (PERSISTENT_CACHE_ROOT / "train").glob("*.npy")
        )),
        "expected": len(train_samples),
    },
    "validation": {
        "present": len(list(
            (PERSISTENT_CACHE_ROOT / "validation").glob("*.npy")
        )),
        "expected": len(validation_samples),
    },
    "trainAudit": train_cache_audit,
    "validationAudit": validation_cache_audit,
    "internalTestAccessed": False,
}
cache_report["complete"] = (
    cache_report["train"]["present"] == cache_report["train"]["expected"]
    and cache_report["validation"]["present"]
    == cache_report["validation"]["expected"]
)

(RUN_ROOT / "persistent_cache_audit.json").write_text(
    json.dumps(cache_report, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(json.dumps(cache_report, indent=2, ensure_ascii=False))

if not cache_report["complete"]:
    raise RuntimeError("Caché incompleto: volver a ejecutar 3A.")


In [ ]:
# 3A-checkpoint) Ejecutar solo después de interrumpir 3A
import time

time.sleep(5)
temporary_files = list(PERSISTENT_CACHE_ROOT.rglob("*.tmp.npy"))
for temporary in temporary_files:
    temporary.unlink(missing_ok=True)

print(json.dumps({
    "cacheRoot": str(PERSISTENT_CACHE_ROOT),
    "persistent": str(PERSISTENT_CACHE_ROOT).startswith("/content/drive/"),
    "trainCached": len(list(
        (PERSISTENT_CACHE_ROOT / "train").glob("*.npy")
    )),
    "trainExpected": len(train_manifest),
    "validationCached": len(list(
        (PERSISTENT_CACHE_ROOT / "validation").glob("*.npy")
    )),
    "validationExpected": len(validation_manifest),
    "temporaryFilesRemoved": len(temporary_files),
    "safeToClose": True,
}, indent=2, ensure_ascii=False))


## 3B — GPU: copiar el caché al SSD local

Después de completar 3A, cambiar a T4 o superior, volver a ejecutar 1 y 2, y luego ejecutar 3B.


In [ ]:
# 3B) Validar caché persistente y copiarlo al SSD local
if not torch.cuda.is_available():
    raise RuntimeError("3B requiere GPU T4 o superior.")

train_samples = prepare_samples(train_manifest, data_root, "train")
validation_samples = prepare_samples(
    validation_manifest,
    data_root,
    "validation",
)

def audit_exact_cache(samples, root: Path, split: str) -> dict:
    expected = set(samples["cache_file"].astype(str))
    actual = {path.name for path in (root / split).glob("*.npy")}
    return {
        "expected": len(expected),
        "present": len(actual),
        "missing": sorted(expected - actual)[:20],
        "unexpected": sorted(actual - expected)[:20],
        "complete": expected == actual,
    }

persistent_train = audit_exact_cache(
    train_samples, PERSISTENT_CACHE_ROOT, "train"
)
persistent_validation = audit_exact_cache(
    validation_samples, PERSISTENT_CACHE_ROOT, "validation"
)

if not (
    persistent_train["complete"]
    and persistent_validation["complete"]
):
    raise RuntimeError({
        "message": "El caché persistente está incompleto.",
        "train": persistent_train,
        "validation": persistent_validation,
    })

if LOCAL_CACHE_ROOT.exists():
    shutil.rmtree(LOCAL_CACHE_ROOT)
LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        "rsync", "-a", "--delete", "--info=progress2",
        f"{PERSISTENT_CACHE_ROOT}/",
        f"{LOCAL_CACHE_ROOT}/",
    ],
    check=True,
)

local_train = audit_exact_cache(train_samples, LOCAL_CACHE_ROOT, "train")
local_validation = audit_exact_cache(
    validation_samples,
    LOCAL_CACHE_ROOT,
    "validation",
)
if not (local_train["complete"] and local_validation["complete"]):
    raise RuntimeError("La copia local del caché quedó incompleta.")

CACHE_ROOT = LOCAL_CACHE_ROOT
print({
    "gpu": torch.cuda.get_device_name(0),
    "trainingCacheRoot": str(CACHE_ROOT),
    "trainFiles": local_train["present"],
    "validationFiles": local_validation["present"],
    "readyForTraining": True,
})


## 4 — Entrenamiento

El mejor checkpoint se selecciona exclusivamente con `validation`. La implementación guarda checkpoints por epoch, pero no reanuda automáticamente el optimizador ni el epoch si esta celda se interrumpe.


In [ ]:
# 4) Entrenar y evaluar sobre validation
if not torch.cuda.is_available():
    raise RuntimeError("La celda 4 requiere GPU.")
if "CACHE_ROOT" not in globals() or CACHE_ROOT != LOCAL_CACHE_ROOT:
    raise RuntimeError("Ejecutá 3B antes de entrenar.")

summary = train_model(
    train_samples=train_samples,
    validation_samples=validation_samples,
    cache_root=CACHE_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    run_root=RUN_ROOT,
    manifest_hashes=manifest_hashes,
    repo_ref=REPO_REF,
    repo_sha=REPO_SHA,
    config=CFG,
)
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
# 5) Gate final y artefactos
required_outputs = [
    "training_history.csv",
    "validation_predictions.csv",
    "validation_metrics_by_group.csv",
    "sampling_audit.json",
    "model_card.md",
    "training_summary.json",
]
missing_outputs = [
    name
    for name in required_outputs
    if not (RUN_ROOT / name).is_file()
]
if missing_outputs:
    raise RuntimeError(f"Faltan outputs: {missing_outputs}")
if summary["approved"] is not True:
    raise RuntimeError(
        "El entrenamiento requiere revisión. "
        "No abrir el internal test ni ajustar gates con él."
    )
if summary["status"] != "APPROVED_FOR_NOTEBOOK_64":
    raise RuntimeError("Estado final inesperado.")
if summary["governance"]["internalTestAccessed"] is not False:
    raise RuntimeError("Se declaró acceso indebido al internal test.")

print({
    "status": summary["status"],
    "bestEpoch": summary["bestEpoch"],
    "validationMetrics": summary["validationMetrics"],
    "checkpoint": summary["checkpoint"],
    "outputs": required_outputs,
    "internalTestSealedUntilNotebook64": True,
    "officialTestAccessed": False,
})
